In [ ]:
# Ignore all other GPUs except this one
!export CUDA_VISIBLE_DEVICES=0

In [ ]:
import numpy as np         
import matplotlib.pyplot as plt     
from matplotlib.animation import FuncAnimation          
import torch      
import torch.nn as nn    
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from torch.utils.data import TensorDataset, DataLoader        
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 
import time      
from scipy.ndimage import uniform_filter1d    
import pandas as pd
import pickle 
import os
from IPython.display import HTML

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
##Plasma Parmeters in normalized units##
n_e = 1.0  # Normalized electron density
T_e = 1.0  # Normalized electron temperature (k_B T_e / eV)
omega_p = 1.0  # Plasma frequency in natural units
v_th = 1.0  # Thermal velocity in natural units
lambda_D = 1.0  # Debye length in natural units

#normalized length and time grid#
L = 10 
nx = 200 #number of spatial poitns
dx = L/nx 
x = np.linspace(0,L,nx)
dt = .01 # time steps in plasma periods omeg_p**-1
nt = 200 #number of time steps

#normalized wavenumber
k=2 * np.pi/L
#Dispersion relation for langmuir waves#
omega = np.sqrt(omega_p**2 + 3*(k**2)*(v_th**2))

In [ ]:
#IC: Small perturbation in electrostatic potential 
phi = np.sin(k*x)
E = -np.gradient(phi, dx)
E_new = np.zeros(nx)
E_time = np.zeros((nt,nx))

# Time loop
for t in range(nt):
    # Update electric field with normalized frequency
    E_new = E * np.cos(omega * t * dt)
    
    # Store electric field for animation
    E_time[t, :] = E_new
    

# Create an animation of the wave
fig, ax = plt.subplots()
line, = ax.plot(x, E_time[0, :])

def update(frame):
    line.set_ydata(E_time[frame, :])
    return line,
plt.title("1D Langmuir Wave")
ani = FuncAnimation(fig, update, frames=nt, interval=50, blit=True)
HTML(ani.to_jshtml())
